## Colab 준비
Colab에서 열었다면 **아래 칸을 맨 먼저 실행**하세요. 도우미 파일과 한글 글꼴을 받아요 (10~20초).

- 더 빠르게: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**
- 고친 내용 저장: **파일 → Drive에 사본 저장**
- 맥 실습실에서 열었다면 이 칸은 아무 일도 안 해요.

In [ ]:
import os, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/sangwon9401/silicon-descent/main/colab/"
    for f in ['common.py', 'solutions/__init__.py', 'solutions/sol00.py', 'solutions/sol01.py', 'solutions/sol02.py', 'solutions/sol04.py']:
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + f, f)
    os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
    print("Colab 준비 완료 ✅")
else:
    print("Colab이 아니에요. 그냥 넘어가요.")

# 00. 실습 준비: 이 실습에서 쓰는 라이브러리

1단계 실습 코드에 나오는 `model`, `tok`, `W`, `cfg`, `run_check`, `x @ w.T`, `einsum` 같은 것들이 **어느 라이브러리의 무엇인지** 하나씩 정리해요.
각 도구마다 ① 설명 → ② 실행 예시 → ③ 작은 연습 문제(자동 채점) 순서예요.

- `...` 이 있는 곳이 채울 부분, 막히면 `# from solutions...` 줄의 `#`을 지우고 실행하면 정답을 불러와요.

## 0. 라이브러리 지도

| 이름 | 어디서 왔나 | 하는 일 | 실습에서 |
|---|---|---|---|
| `torch` | **PyTorch** | 텐서(숫자 상자)와 그 계산 전부. GPU(cuda/mps)에서 돌려 줌 | 모든 계산 |
| `F` (`torch.nn.functional`) | PyTorch | 활성화 함수 같은 '상태 없는 함수' 모음 | `F.silu` |
| `transformers` | **HuggingFace** | 공개 모델을 내려받아 바로 돌려 주는 라이브러리. 모델(`AutoModelForCausalLM`)과 토크나이저(`AutoTokenizer`) | `model`, `tok` |
| `common` | **이 코스 전용** (`labs/common.py`) | HF 모델을 불러오고, 가중치를 꺼내 짧은 이름으로 정리하고, 채점해 주는 도우미 | `load_model`, `model_cfg`, `extract_weights`, `run_check`, `check_close`, `pick_device`, `sync`, `timeit` |
| `solutions` | 이 코스 전용 (`labs/solutions/`) | 정답 코드 | 막혔을 때 |
| `matplotlib` | 그래프 라이브러리 | 측정 결과 그리기 | 02, 03번 |

핵심 흐름: **transformers로 모델을 받고 → common이 그 안의 가중치 텐서를 꺼내 W에 담고 → 우리는 torch로 그 가중치를 직접 곱해서 같은 답을 만든다.**

In [ ]:
import torch
import torch.nn.functional as F
from common import *

device = pick_device()
print("torch 버전:", torch.__version__, "/ 장치:", device)

---
## 1. HuggingFace transformers: `model`과 `tok`

`load_model(device)`는 `common.py` 안에서 사실상 이 두 줄을 해요.
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
tok   = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M").to(device).eval()
```
- `from_pretrained(이름)`: HuggingFace Hub에서 모델 파일(설정 `config.json`, 가중치 `model.safetensors`, 토크나이저 파일)을 내려받아요. 한 번 받으면 `~/.cache/huggingface`에 저장돼서 다음엔 바로 열려요.
- `Auto...`: 설정 파일을 보고 맞는 모델 종류(여기선 Llama 구조)를 알아서 골라 줘요.
- `.eval()`: 학습 모드가 아니라 추론 모드로. `.to(device)`: 가중치를 GPU로 옮기기.

처음 실행하면 약 270MB를 받아서 1~2분 걸려요.

In [ ]:
model, tok = load_model(device)
print(model)

### `print(model)` 읽는 법
위 출력이 모델의 **부품 트리**예요. 0단계에서 배운 그대로예요.
```
LlamaForCausalLM
  model
    embed_tokens        ← 임베딩 표 [49152, 576]
    layers: 30 x LlamaDecoderLayer      ← 블록 30개
      self_attn: q_proj, k_proj, v_proj, o_proj   ← Wq, Wk, Wv, Wo
      mlp: gate_proj, up_proj, down_proj          ← SwiGLU 표 3개
      input_layernorm, post_attention_layernorm   ← RMSNorm 두 개
    norm                ← 마지막 RMSNorm
  lm_head               ← 다음 토큰 점수 표
```
부품은 **점(.)으로 따라 들어가며** 꺼내요. 숫자 표(텐서)는 `.weight`에 들어 있어요.
예: 블록 0의 Wq → `model.model.layers[0].self_attn.q_proj.weight`

모델 설정값은 `model.config`에 있어요.

In [ ]:
print("블록 0의 Wq 모양:", model.model.layers[0].self_attn.q_proj.weight.shape)
print("레이어 수:", model.config.num_hidden_layers, "/ Q 헤드:", model.config.num_attention_heads,
      "/ KV 헤드:", model.config.num_key_value_heads, "/ hidden:", model.config.hidden_size)
print("전체 파라미터 수:", sum(p.numel() for p in model.parameters()))

### 연습 1-1: HF 모델 안에서 가중치 꺼내기
블록 `i`의 **k_proj 가중치 텐서**를 돌려주세요. 힌트: `model.model.layers[i].self_attn.k_proj.weight`

In [ ]:
def hf_k_weight(model, i):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import hf_k_weight

In [ ]:
run_check("연습 1-1", lambda: check_close("k_proj 꺼내기", hf_k_weight(model, 2), model.model.layers[2].self_attn.k_proj.weight))

### 토크나이저 `tok`

| 코드 | 뜻 |
|---|---|
| `tok("The capital", return_tensors="pt")` | 문장 → 결과 묶음(딕셔너리 비슷). `return_tensors="pt"`는 'PyTorch 텐서로 줘' |
| `.input_ids` | 묶음에서 토큰 번호만 → 모양 **[1, T]** (문장 1개 × 토큰 T개). 묶음엔 `attention_mask`도 있지만 실습에선 안 써요 |
| `[0]` | 첫 문장만 → **[T]** |
| `tok.decode([번호들])` | 번호 → 글자 |

그래서 01번의 `ids = tok(prompt, return_tensors="pt").input_ids[0].to(device)` = "토큰 번호 [T]를 만들어 계산 장치로".

In [ ]:
enc = tok("The capital of France is", return_tensors="pt")
print("묶음 안:", list(enc.keys()))
ids = enc.input_ids[0].to(device)
print("ids:", ids.tolist(), " 모양", tuple(ids.shape))
print("조각:", [tok.decode([i]) for i in ids])

### 모델 실행하기: `model(...)`
- HF 모델은 **문장 묶음(batch)**을 받아요. 그래서 [T]가 아니라 **[1, T]**로 넣어요: `ids[None]` (맨 앞에 차원 하나 추가)
- 결과는 객체예요. 점수표는 `.logits` → 모양 **[1, T, 49152]** = 문장 1개 × 토큰 위치 T개 × 어휘 전체 점수
- 다음 토큰 = 첫 문장(`[0]`)의 마지막 위치(`-1`)에서 가장 큰 점수의 번호: `logits[0, -1].argmax()`
- `with torch.no_grad():` 안에서 돌리면 학습용 기록(기울기)을 안 남겨서 빠르고 메모리를 아껴요.

In [ ]:
with torch.no_grad():
    out = model(ids[None])
print("logits 모양:", tuple(out.logits.shape))
nxt = out.logits[0, -1].argmax()
print("다음 토큰:", nxt.item(), repr(tok.decode([nxt.item()])))

### 연습 1-2: HF 모델로 다음 토큰 번호 구하기
`ids`는 [T]. 힌트: `model(ids[None]).logits[0, -1].argmax()` (no_grad 안에서)

In [ ]:
def hf_next_token(model, ids):
    with torch.no_grad():
        ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import hf_next_token

In [ ]:
run_check("연습 1-2", lambda: check_close("HF 다음 토큰", hf_next_token(model, ids), nxt))

---
## 2. 이 코스의 도우미 `common.py`

HF 모델은 부품 이름이 길어서(`model.model.layers[0].self_attn.q_proj.weight`), 직접 짜기 편하게 **텐서만 꺼내 짧은 이름으로** 정리했어요.

**`cfg = model_cfg(model)`** → 필요한 설정만 담은 딕셔너리

| 키 | 뜻 | SmolLM2 |
|---|---|---|
| `L` | 레이어 수 | 30 |
| `nq` / `nkv` | Q 헤드 수 / KV 헤드 수 | 9 / 3 |
| `hd` | 헤드 하나의 크기 | 64 |
| `eps` | RMSNorm에서 0으로 나누기 방지용 작은 수 | 1e-5 |
| `theta` | RoPE 회전 속도 기준값 | 100000 |

**`W = extract_weights(model)`** → 가중치 텐서 딕셔너리

| W의 이름 | HF 원래 이름 | 모양 |
|---|---|---|
| `W["embed"]` | `model.model.embed_tokens.weight` | [49152, 576] |
| `W["layers"][i]["ln1"]`, `["ln2"]` | `layers[i].input_layernorm.weight`, `post_attention_layernorm.weight` | [576] |
| `W["layers"][i]["wq"]`, `["wk"]`, `["wv"]`, `["wo"]` | `layers[i].self_attn.q_proj/k_proj/v_proj/o_proj.weight` | [576,576], [192,576], [192,576], [576,576] |
| `W["layers"][i]["w_gate"]`, `["w_up"]`, `["w_down"]` | `layers[i].mlp.gate_proj/up_proj/down_proj.weight` | [1536,576], [1536,576], [576,1536] |
| `W["norm"]` | `model.model.norm.weight` | [576] |
| `W["lm_head"]` | `model.lm_head.weight` | [49152, 576] |

같은 텐서를 가리키는 **다른 이름**일 뿐이라, 값이 완전히 같아요. 모양이 **[출력, 입력]**인 것도 기억해 두세요(그래서 `x @ w.T`).

In [ ]:
cfg = model_cfg(model)
W = extract_weights(model)
print("cfg:", cfg)
for name, w in W["layers"][0].items():
    print(f"W['layers'][0]['{name}']".ljust(28), tuple(w.shape))
print("같은 텐서?", torch.equal(W["layers"][0]["wq"], model.model.layers[0].self_attn.q_proj.weight))

### 채점 도우미
- `check_close(이름, 내 결과, 정답)`: 두 텐서의 모양과 값이 거의 같은지 보고 ✅/❌를 출력해요.
- `check_equal(이름, 내 결과, 정답)`: 정확히 같은지(토큰 목록 비교용).
- `run_check(이름, lambda: check_...)`: 채점을 대신 실행하면서, 내 함수에서 에러가 나도 멈추지 않고 친절한 ❌ 메시지로 바꿔 줘요. `lambda:`는 '이 코드를 바로 실행하지 말고 run_check 안에서 실행해 줘'라는 뜻이에요.
- 그 밖에: `pick_device()` 장치 고르기, `sync(device)` GPU 작업 끝날 때까지 기다리기(시간 잴 때), `timeit(fn, device)` 평균 실행 시간.

### 연습 2-1: W에서 꺼내기
블록 `i`의 Wk를 **W에서** 꺼내 돌려주세요. 힌트: `W["layers"][i]["wk"]`

### 연습 2-2: cfg로 계산하기
K 벡터 전체 크기(KV 헤드 수 × 헤드 크기)를 cfg로 계산해 돌려주세요. 힌트: `cfg["nkv"] * cfg["hd"]` → 192

In [ ]:
def get_wk(W, i):
    ...  # 여기를 채우세요


def kv_dim(cfg):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import get_wk, kv_dim

In [ ]:
run_check("연습 2-1", lambda: check_close("W에서 Wk", get_wk(W, 5), model.model.layers[5].self_attn.k_proj.weight))
run_check("연습 2-2", lambda: check_equal("KV 크기", kv_dim(cfg), model.config.num_key_value_heads * model.config.head_dim))

---
# 3. PyTorch 텐서 도구
이제 W 안의 텐서를 직접 계산하는 데 쓰는 PyTorch 도구들이에요. 01번 실습의 빈칸은 전부 이 도구의 조합이에요.

---
## 3-1. 텐서: 숫자가 담긴 상자

PyTorch의 기본 단위는 **텐서(tensor)**예요. 숫자를 여러 차원으로 담은 상자예요.
- 숫자 하나 → 0차원, 줄 하나 `[1, 2, 3]` → 1차원(벡터), 표 → 2차원(행렬), 표 여러 장 → 3차원

**`.shape`(모양)이 제일 중요해요.** 실습에서 막히면 거의 항상 모양 문제예요. 막히면 `print(x.shape)`부터!

| 만들기 | 뜻 |
|---|---|
| `torch.tensor([1., 2.])` | 직접 숫자 넣기 |
| `torch.randn(3, 4)` | 무작위 숫자로 3×4 |
| `torch.zeros(3, 4)`, `torch.ones(3, 4)` | 0 또는 1로 채운 3×4 |
| `torch.arange(5)` | `[0, 1, 2, 3, 4]` |
| `torch.arange(2, 5)` | `[2, 3, 4]` (끝은 안 들어가요) |

꺼내기: `.tolist()` → 파이썬 리스트, `.item()` → 숫자 하나짜리 텐서를 그냥 숫자로.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.randn(3, 4)
print("a:", a, "모양", a.shape)
print("b 모양:", b.shape, " → 3줄 4칸")
print("arange(5):", torch.arange(5))
print("tolist:", torch.arange(3).tolist(), "/ item:", torch.tensor(7).item())

# reshape: 숫자 순서는 그대로, 모양만 바꾸기
g = torch.arange(6)
print("arange(6).reshape(2, 3):\n", g.reshape(2, 3))

### device: 어느 장치에서 계산하나
맥에서는 `mps`(맥 GPU), NVIDIA GPU면 `cuda`, 없으면 `cpu`.
계산하는 텐서들은 **같은 장치에 있어야** 해요. 옮길 때는 `.to(device)`.
다르면 `Expected all tensors to be on the same device` 에러가 나요.

In [ ]:
device = pick_device()
c = torch.randn(2, 2).to(device)
print("device:", device, "/ c는 어디에?", c.device)

### 연습 3-1: 0부터 11까지를 3×4 표로
힌트: `torch.arange(12).reshape(3, 4)`

In [ ]:
def make_grid():
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import make_grid

In [ ]:
run_check("연습 3-1", lambda: check_close("3×4 표", make_grid(), torch.arange(12).reshape(3, 4)))

---
## 3-2. 꺼내고 자르기 (인덱싱, 슬라이싱)

대괄호 안에 **차원마다** 무엇을 꺼낼지 쉼표로 적어요.

| 코드 | 뜻 (x가 [3, 4] 표일 때) |
|---|---|
| `x[0]` | 0번 줄 → 모양 [4] |
| `x[-1]` | 마지막 줄 |
| `x[-2:]` | 뒤에서 2줄 → [2, 4] |
| `x[:, 1]` | 모든 줄의 1번 칸 → [3]. `:`는 '전부' |
| `x[:, :2]` | 모든 줄의 앞 2칸 → [3, 2] |
| `x[..., :2]` | `...` = '앞 차원들은 전부'. 차원이 몇 개든 **마지막 차원**만 자를 때 써요 |
| `x[:, None, :]` | 가운데에 크기 1인 차원 끼우기 → [3, 1, 4] |

`a:b`는 a부터 b **직전**까지예요. `:half`는 처음부터 half 직전까지, `half:`는 half부터 끝까지.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print("x =\n", x)
print("x[0] =", x[0])
print("x[-2:] =\n", x[-2:])
print("x[:, 1] =", x[:, 1])
print("x[..., :2] =\n", x[..., :2])
print("x[:, None, :] 모양:", x[:, None, :].shape)

### 연습 3-2: 뒤 2줄 / 1번 칸
힌트: `x[-2:]`, `x[:, 1]`

In [ ]:
def last_two_rows(x):
    ...  # 여기를 채우세요


def second_col(x):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import last_two_rows, second_col

In [ ]:
_t = torch.arange(20).reshape(5, 4)
run_check("연습 3-2-1", lambda: check_close("뒤 2줄", last_two_rows(_t), _t[-2:]))
run_check("연습 3-2-2", lambda: check_close("1번 칸", second_col(_t), _t[:, 1]))

### torch.cat: 이어 붙이기
`torch.cat([a, b], dim=-1)` → a 뒤에 b를 **마지막 차원 방향**으로 붙여요. `dim=0`이면 아래로(줄 방향) 붙여요.
`-x`는 모든 숫자의 부호를 바꿔요.

In [ ]:
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[5, 6], [7, 8]])
print("dim=-1 (옆으로):\n", torch.cat([a, b], dim=-1))
print("dim=0 (아래로):\n", torch.cat([a, b], dim=0))

### 연습 3-3: 마지막 차원의 앞 절반과 뒤 절반 바꾸기
`[1, 2, 3, 4]` → `[3, 4, 1, 2]`. 차원이 몇 개든 마지막 차원에 대해서요.

힌트
- `half = x.shape[-1] // 2` (`//`는 몫만 남기는 나눗셈)
- 앞 절반 `x[..., :half]`, 뒤 절반 `x[..., half:]`
- `torch.cat([뒤, 앞], dim=-1)`

(01번 실습의 `rotate_half`가 이것과 아주 비슷해요!)

In [ ]:
def swap_halves(x):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import swap_halves

In [ ]:
_r = torch.randn(3, 2, 8)
run_check("연습 3-3", lambda: check_close("절반 바꾸기", swap_halves(_r), torch.cat([_r[..., 4:], _r[..., :4]], dim=-1)))

---
## 3-3. 계산하기

**원소별 계산**: `+ - * /`, `x.pow(2)`(제곱), `x.sqrt()`, `torch.rsqrt(x)`(1/√x). 같은 자리끼리 계산해요.

**줄이기**: `x.sum(dim=-1)`, `x.mean(dim=-1)` → 마지막 차원을 합/평균으로 **없애요**. [3, 4] → [3]
- `keepdim=True`를 붙이면 없애지 않고 크기 1로 남겨요. [3, 4] → [3, 1]. 그래야 원래 [3, 4]와 나눌 수 있어요.

**브로드캐스팅**: 모양이 달라도 크기 1인 차원은 자동으로 늘려서 계산해 줘요.
[3, 4] ÷ [3, 1] → [3, 1]이 4칸으로 복사된 것처럼 계산돼요.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0], [2.0, 2.0, 2.0, 2.0]])
print("x.pow(2) =\n", x.pow(2))
print("mean(dim=-1) =", x.mean(dim=-1), " 모양", x.mean(dim=-1).shape)
m = x.mean(dim=-1, keepdim=True)
print("keepdim=True 모양:", m.shape)
print("x / m (브로드캐스팅) =\n", x / m)

### 연습 3-4: 줄마다 제곱의 평균 (keepdim 유지)
모양 [T, D] → [T, 1]. 01번 실습 RMSNorm의 첫 단계예요.

힌트: `x.pow(2).mean(dim=-1, keepdim=True)`

In [ ]:
def row_mean_sq(x):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import row_mean_sq

In [ ]:
_x = torch.randn(5, 576)
run_check("연습 3-4", lambda: check_close("제곱 평균", row_mean_sq(_x), (_x ** 2).mean(-1, keepdim=True)))

---
## 3-4. 행렬곱 `@`와 `.T`

`a @ b`는 행렬곱이에요(0단계 3번 강의). **a의 마지막 크기 = b의 첫 크기**여야 해요. [T, 576] @ [576, 192] → [T, 192]

HuggingFace 가중치는 모양이 **[출력, 입력]**으로 저장돼 있어요. 예: Wk는 [192, 576].
그래서 x [T, 576]에 곱하려면 뒤집어서(`.T`, 전치) [576, 192]로 만들어 `x @ w.T`로 곱해요.

모양이 안 맞으면 이런 에러가 나요: `mat1 and mat2 shapes cannot be multiplied (5x576 and 192x576)` → `.T`를 빼먹은 거예요!

In [ ]:
x = torch.randn(5, 576)
w = torch.randn(192, 576)   # [출력 192, 입력 576]
print("w.T 모양:", w.T.shape)
print("x @ w.T 모양:", (x @ w.T).shape)

### 연습 3-5: 선형 층
x [T, in], w [out, in] → [T, out]

In [ ]:
def linear(x, w):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import linear

In [ ]:
run_check("연습 3-5", lambda: check_close("선형 층", linear(x, w), x @ w.T, atol=1e-3))

---
## 3-5. 모양 바꾸기: view, reshape, repeat_interleave

`x.view(T, 9, 64)` → 숫자 순서는 그대로 두고 모양만 [T, 576]에서 [T, 9, 64]로. **576 = 9 × 64**라서 가능해요.
576칸을 '64칸씩 9묶음(헤드 9개)'으로 보는 거예요(0단계 6번 멀티헤드).

`-1`을 쓰면 '나머지는 알아서 계산해'예요. `x.view(T, 9, -1)` → 마지막은 64로 자동.
`reshape`는 `view`와 거의 같은데 더 너그러워요(메모리가 흩어져 있어도 동작). 헷갈리면 `reshape`를 써도 돼요.

`k.repeat_interleave(3, dim=1)` → 1번 차원의 각 칸을 3번씩 복사해요. [S, 3, 64] → [S, 9, 64]. GQA에서 KV 헤드를 Q 헤드 수에 맞출 때 써요.

In [ ]:
x = torch.arange(12).reshape(2, 6)
print("view(2, 3, 2):\n", x.view(2, 3, 2))
h = torch.tensor([[10, 20, 30]])              # [1, 3]
print("repeat_interleave(2, dim=1):", h.repeat_interleave(2, dim=1))   # 10,10,20,20,30,30

### 연습 3-6: 헤드로 나누기 / 헤드 복사하기
- `split_heads(x, n_heads)`: [T, D] → [T, n_heads, D / n_heads]. 힌트: `x.view(T, n_heads, -1)`
- `repeat_heads(k, times)`: [S, n, d] → [S, n × times, d]. 힌트: `k.repeat_interleave(times, dim=1)`

In [ ]:
def split_heads(x, n_heads):
    T = x.shape[0]
    ...  # 여기를 채우세요


def repeat_heads(k, times):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import split_heads, repeat_heads

In [ ]:
_h = torch.randn(4, 576)
_k = torch.randn(4, 3, 64)
run_check("연습 3-6-1", lambda: check_close("헤드 나누기", split_heads(_h, 9), _h.view(4, 9, 64)))
run_check("연습 3-6-2", lambda: check_close("헤드 복사", repeat_heads(_k, 3), _k.repeat_interleave(3, dim=1)))

---
## 3-6. einsum: 이름표로 쓰는 곱셈

`torch.einsum("규칙", a, b)`는 **차원마다 글자 이름을 붙여서** 곱하는 방법을 적어요.

읽는 법 3가지
1. 쉼표 앞은 a의 차원 이름, 뒤는 b의 차원 이름, 화살표 뒤는 결과의 차원 이름.
2. a와 b에 **같은 글자**가 있으면 그 차원끼리 짝지어 곱해요.
3. 화살표 뒤에 **없는 글자**는 더해서 없애요.

예: `"ij,jk->ik"` = 보통 행렬곱 `a @ b`. j가 결과에 없으니 j 방향으로 곱해서 더해요(내적).

01번 실습의 `"tnd,snd->nts"`
- q는 [t(내 토큰), n(헤드), d(64칸)], k는 [s(상대 토큰), n, d]
- n은 양쪽에 있고 결과에도 있음 → 헤드끼리만 짝지음(헤드마다 따로)
- d는 결과에 없음 → 64칸을 곱해서 더함 = **q·k 내적**
- 결과 [n, t, s] = 헤드마다, 내 토큰 t가 상대 토큰 s를 보는 점수표

In [ ]:
a = torch.randn(2, 3)
b = torch.randn(3, 4)
print("einsum ij,jk->ik 와 a @ b 가 같나?", torch.allclose(torch.einsum("ij,jk->ik", a, b), a @ b))

q = torch.randn(5, 9, 64)   # 토큰 5개, 헤드 9개
k = torch.randn(5, 9, 64)
print("점수표 모양 [헤드, 내 토큰, 상대 토큰]:", torch.einsum("tnd,snd->nts", q, k).shape)

### 연습 3-7: 헤드마다 점수표
q [T, n, d], k [S, n, d] → [n, T, S]

In [ ]:
def head_scores(q, k):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import head_scores

In [ ]:
_q, _k2 = torch.randn(3, 2, 8), torch.randn(4, 2, 8)
run_check("연습 3-7", lambda: check_close("점수표", head_scores(_q, _k2), (_q.transpose(0, 1) @ _k2.permute(1, 2, 0))))

---
## 3-7. 비교, masked_fill, softmax

**비교하면 참/거짓 텐서가 나와요.** `torch.arange(4) > 1` → `[False, False, True, True]`

**브로드캐스팅으로 표 만들기**: `[:, None]`(세로)와 `[None, :]`(가로)를 비교하면 모든 짝의 비교표가 생겨요.
`qpos = arange(T)[:, None]`, `kpos = arange(T)[None, :]` → `kpos > qpos`는 '상대가 나보다 뒤(미래)'인 칸만 True인 표예요.

**`x.masked_fill(조건, 값)`**: 조건이 True인 칸을 값으로 바꿔요. 미래 칸을 `float("-inf")`(마이너스 무한대)로 → softmax 후 0%가 돼요(0단계 5번).

**`x.softmax(dim=-1)`**: 마지막 차원을 합이 1인 비율로(0단계 4번).

In [ ]:
T = 4
qpos = torch.arange(T)[:, None]
kpos = torch.arange(T)[None, :]
future = kpos > qpos
print("미래인 칸 (True):\n", future)
s = torch.zeros(T, T).masked_fill(future, float("-inf"))
print("가린 점수:\n", s)
print("softmax:\n", s.softmax(dim=-1))

### 연습 3-8: 정사각형 점수표의 미래 가리기
scores [..., T, T]에서 '상대가 나보다 뒤'인 칸을 `-inf`로.

In [ ]:
def causal_mask(scores):
    T = scores.shape[-1]
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import causal_mask

In [ ]:
_s = torch.randn(2, 5, 5)
run_check("연습 3-8", lambda: check_close("미래 가리기", causal_mask(_s), _s.masked_fill(torch.ones(5, 5, dtype=torch.bool).triu(1), float("-inf"))))

---
## 3-8. 나머지 도구들

| 도구 | 뜻 | 어디서 |
|---|---|---|
| `F.silu(x)` | SiLU 활성화 (0단계 10번) | 01 MLP |
| `x.argmax()` / `x.argmax(dim=-1)` | 가장 큰 값의 **위치(번호)** | 다음 토큰 고르기 |
| `logits[-1]` | 마지막 토큰의 점수 | 다음 토큰 고르기 |
| `with torch.no_grad():` | 학습 안 하니 기울기 기록 끄기 (빠르고 메모리 절약) | 02 |
| `x.sort(dim=-1, descending=True)` | 정렬. (값, 원래 위치) 두 개를 돌려줘요 | 04 top-k/p |
| `x.cumsum(dim=-1)` | 누적 합 [1, 2, 3] → [1, 3, 6] | 04 top-p |
| `torch.where(조건, a, b)` | 조건이 참이면 a, 아니면 b | 04 |
| `torch.multinomial(probs, 1)` | 확률대로 하나 뽑기 | 04 |
| `x.clone()` | 복사본 만들기 | 01 |

In [ ]:
logits = torch.tensor([[0.1, 2.0, 0.5], [3.0, 0.2, 1.0]])
print("마지막 줄:", logits[-1], "→ argmax:", logits[-1].argmax().item())
v, i = torch.tensor([3.0, 1.0, 2.0]).sort(descending=True)
print("sort 값:", v, "원래 위치:", i)
print("cumsum:", torch.tensor([1, 2, 3]).cumsum(dim=-1))
print("where:", torch.where(torch.tensor([True, False]), torch.tensor([1, 1]), torch.tensor([9, 9])))

### 연습 3-9: 다음 토큰 번호
logits [T, V]에서 **마지막 토큰** 줄의 가장 큰 값 위치. 힌트: `logits[-1].argmax()`

In [ ]:
def next_token(logits):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import next_token

In [ ]:
_l = torch.randn(6, 100)
run_check("연습 3-9", lambda: check_close("다음 토큰", next_token(_l), _l[-1].argmax()))

---
## 🎉 다 했어요!

막혔을 때 쓰는 3단계 습관
1. **`print(x.shape)`** 로 모양부터 확인해요. 에러의 대부분은 모양이 안 맞아서예요.
2. 에러 메시지의 **마지막 줄**을 읽어요. `shapes cannot be multiplied` → `.T` 확인, `same device` → `.to(device)` 확인, `NoneType` → `return` 빠뜨림.
3. 작은 숫자로 따로 실험해 봐요. 새 셀을 만들어 `torch.arange(6).reshape(2, 3)` 같은 걸로 시험해 보면 금방 보여요.

이제 **01번 실습**으로 가세요. 거기 나오는 코드가 이 노트북의 도구 조합이에요.